In [44]:
# 1. Setup and Imports
import os
import json
import random
from typing import List, Dict, Any, Optional
from pathlib import Path

# LangChain and Google Gemini
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

# Progress tracking
from tqdm.auto import tqdm

# Set random seed for reproducibility
RANDOM_SEED = 67
random.seed(RANDOM_SEED)

# Configure paths - UPDATED FOR VQA (PLANTWILD)
METADATA_FILE = Path("data/vqa/samples.json")
REFERENCE_FILE = Path("data/vqa/reference.json")
OUTPUT_DIR = Path("data/vqa/")
OUTPUT_FILE = OUTPUT_DIR / "vqa_dataset.json"
SCENARIOS_FILE = OUTPUT_DIR / "vqa_all_scenarios_id.json"

print(f"Metadata file: {METADATA_FILE}")
print(f"Reference file: {REFERENCE_FILE}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Scenarios file: {SCENARIOS_FILE}")
print(f"Random seed: {RANDOM_SEED}")

# Load existing results if they exist to avoid re-generating
existing_results_lookup = {}
if SCENARIOS_FILE.exists():
    try:
        with open(SCENARIOS_FILE, 'r') as f:
            existing_data = json.load(f)
            for entry in existing_data:
                if "metadata" in entry and "filename" in entry["metadata"]:
                    filename = entry["metadata"]["filename"]
                    existing_results_lookup[filename] = entry
        print(f"✅ Loaded {len(existing_results_lookup)} existing entries from {SCENARIOS_FILE.name}")
    except Exception as e:
        print(f"⚠️ Could not load existing scenarios: {e}")
else:
    print(f"No existing scenarios file found at {SCENARIOS_FILE}")


Metadata file: data\vqa\samples.json
Reference file: data\vqa\reference.json
Output file: data\vqa\vqa_dataset.json
Scenarios file: data\vqa\vqa_all_scenarios_id.json
Random seed: 67
✅ Loaded 266 existing entries from vqa_all_scenarios_id.json


In [45]:
# 2. Define Pydantic Models for Structured Output
class VQAPair(BaseModel):
    """Structured output for VQA pair generation with quality control."""
    
    # Quality control - NEW FIELD
    is_plant_related: bool = Field(
        description="Indicates if the image is related to plant disease/health. Set to FALSE for unrelated images like ornaments, decorations, or non-plant objects"
    )
    
    # Image composition analysis - NEW FIELDS
    image_shows_multiple_leaves: bool = Field(
        description="TRUE if image shows multiple leaves clearly visible (2 or more leaves). FALSE if single leaf or no leaves"
    )
    image_shows_non_leaf_parts: bool = Field(
        description="TRUE if image primarily shows non-leaf plant parts like fruits, stems, flowers, pods, or roots. FALSE if primarily showing leaves"
    )
    
    user_text: str = Field(
        description="The user's question or statement about the plant image"
    )
    reference_answer: str = Field(
        description="The correct answer that the agent should provide"
    )
    reference_goal: str = Field(
        description="High-level objective of the agent based on user text and system prompt"
    )
    reference_context: str = Field(
        description="Detailed context and evidence for the answer"
    )
    reference_tool_calls: List[Dict[str, Any]] = Field(
        description="List of tool calls the agent should make"
    )
    
    # Metadata
    plant: str = Field(description="Plant species name")
    class_name: str = Field(description="Disease or health class name")
    pathogen_type: str = Field(description="Type of pathogen (fungal, viral, bacterial, healthy, etc.)")
    prompt_type: str = Field(description="Scenario type: vague_symptoms, direct_inquiry, general_inquiry")
    filename: str = Field(description="Image filename")

# Print model information after class definition

print(f"Fields: {list(VQAPair.model_fields.keys())}")

Fields: ['is_plant_related', 'image_shows_multiple_leaves', 'image_shows_non_leaf_parts', 'user_text', 'reference_answer', 'reference_goal', 'reference_context', 'reference_tool_calls', 'plant', 'class_name', 'pathogen_type', 'prompt_type', 'filename']


In [46]:
# 3. Load Data and Reference Information
def load_metadata() -> Dict[str, Dict[str, Any]]:
    """Load the metadata.json file."""
    with open(METADATA_FILE, 'r') as f:
        return json.load(f)

def load_reference() -> List[Dict[str, Any]]:
    """Load the reference.json file."""
    with open(REFERENCE_FILE, 'r') as f:
        return json.load(f)

def extract_plant_from_class(class_name: str) -> str:
    """Extract plant name from class name (e.g., 'apple black rot' -> 'apple')."""
    class_name = class_name.replace('_', ' ')
    if class_name.startswith("bell pepper"):
        return "bell pepper"
    return class_name.split()[0]

def extract_pathogen_type(status: str, class_name: str = "") -> str:
    """Determine pathogen type from status and class name."""
    if status == "healthy":
        return "healthy"
    
    # Extract from class name or use common patterns
    if "viral" in class_name.lower() or "virus" in class_name.lower():
        return "viral"
    elif "bacterial" in class_name.lower():
        return "bacterial"
    elif "fungal" in class_name.lower() or any(x in class_name.lower() for x in ["anthracnose", "mildew", "rust", "canker", "spot", "rot"]):
        return "fungal"
    elif "pest" in class_name.lower() or "mite" in class_name.lower() or "insect" in class_name.lower():
        return "pest"
    elif "nutritional deficiency" in class_name.lower():
        return "nutritional deficiency"
    else:
        return "N/A"

# Load data
metadata = load_metadata()
reference_data = load_reference()

# Create reference lookup
reference_lookup = {item["class"]: item for item in reference_data}

print(f"✅ Loaded {len(metadata)} metadata entries")
print(f"✅ Loaded {len(reference_data)} reference entries")

# Validate metadata structure and calculate expected output
def validate_metadata_structure(metadata):
    """Validate metadata and calculate expected dataset size."""
    print(f"\n📋 Metadata Validation:")
    
    # Count by status
    status_counts = {}
    class_status = {}
    for filename, item in metadata.items():
        status = item["status"]
        class_name = item["class"]
        status_counts[status] = status_counts.get(status, 0) + 1
        class_status[class_name] = status
    
    print(f"  Total entries: {len(metadata)}")
    print(f"  By status: {status_counts}")
    
    # Count unique classes
    unique_classes = set(item["class"] for item in metadata.values())
    disease_classes = set(class_name for class_name, status in class_status.items() if status == "diseased")
    healthy_classes = set(class_name for class_name, status in class_status.items() if status == "healthy")
    
    print(f"  Unique classes: {len(unique_classes)}")
    print(f"  Disease classes: {len(disease_classes)}")
    print(f"  Healthy classes: {len(healthy_classes)}")
    
    # Calculate expected output
    disease_images = status_counts.get("diseased", 0)
    healthy_images = status_counts.get("healthy", 0)
    
    # Each image gets 1 scenario, but each class has 3 images = 3 scenarios total per class
    expected_disease_entries = disease_images  # 1 scenario per image
    expected_healthy_entries = healthy_images  # 1 scenario per image
    total_expected = expected_disease_entries + expected_healthy_entries
    
    print(f"\n📊 Expected Dataset Size:")
    print(f"  Disease images: {disease_images} × 1 scenario each = {expected_disease_entries} entries")
    print(f"  Healthy images: {healthy_images} × 1 scenario each = {expected_healthy_entries} entries")
    print(f"  Total expected: {total_expected} entries")
    
    return {
        "disease_classes": len(disease_classes),
        "healthy_classes": len(healthy_classes),
        "disease_images": disease_images,
        "healthy_images": healthy_images,
        "expected_total": total_expected
    }


# Validate and show calculationprint(f"\n✅ Sample classes: {list(metadata.keys())[:5]}")
validation = validate_metadata_structure(metadata)

✅ Loaded 267 metadata entries
✅ Loaded 89 reference entries

📋 Metadata Validation:
  Total entries: 267
  By status: {'diseased': 177, 'healthy': 90}
  Unique classes: 89
  Disease classes: 59
  Healthy classes: 30

📊 Expected Dataset Size:
  Disease images: 177 × 1 scenario each = 177 entries
  Healthy images: 90 × 1 scenario each = 90 entries
  Total expected: 267 entries


In [47]:
# Print detail unique classes untuk verifikasi distribusi
disease_classes = sorted(list(set(item["class"] for item in metadata.values() if item["status"] == "diseased")))
healthy_classes = sorted(list(set(item["class"] for item in metadata.values() if item["status"] == "healthy")))

print(f"📋 JUMLAH KELAS PENYAKIT (DISEASED): {len(disease_classes)}")
for i, cls in enumerate(disease_classes, 1):
    print(f"  {i:2}. {cls}")

print(f"\n📋 JUMLAH KELAS SEHAT (HEALTHY): {len(healthy_classes)}")
for i, cls in enumerate(healthy_classes, 1):
    print(f"  {i:2}. {cls}")


📋 JUMLAH KELAS PENYAKIT (DISEASED): 59
   1. apple black rot
   2. apple mosaic virus
   3. apple rust
   4. apple scab
   5. banana panama disease
   6. basil downy mildew
   7. bean halo blight
   8. bean mosaic virus
   9. bean rust
  10. bell pepper leaf spot
  11. blueberry rust
  12. broccoli downy mildew
  13. cabbage alternaria leaf spot
  14. carrot cavity spot
  15. cauliflower alternaria leaf spot
  16. celery anthracnose
  17. celery early blight
  18. cherry leaf spot
  19. cherry powdery mildew
  20. citrus canker
  21. citrus greening disease
  22. coffee leaf rust
  23. corn gray leaf spot
  24. corn northern leaf blight
  25. corn rust
  26. corn smut
  27. cucumber angular leaf spot
  28. cucumber bacterial wilt
  29. cucumber powdery mildew
  30. eggplant cercospora leaf spot
  31. garlic leaf blight
  32. garlic rust
  33. ginger leaf spot
  34. ginger sheath blight
  35. grape black rot
  36. grape downy mildew
  37. grape leaf spot
  38. grapevine leafroll disease

In [ ]:
# 4. Enhanced Prompt Templates with Quality Control
def get_vague_symptoms_prompt(plant: str, symptoms: str, image_url: str) -> str:
    """Prompt untuk skenario 1: Gejala samar + spesies + permintaan penanganan."""
    return f"""Anda adalah pemilik tanaman yang bukan ahli penyakit tanaman. Anda telah mengambil foto tanaman {plant} Anda dan menyadari adanya masalah.

## CEK KUALITAS KRITIS - BACA DENGAN TELITI:
Anda HARUS mengevaluasi apakah gambar tersebut benar-benar terkait dengan penyakit/kesehatan tanaman:

### TERIMA gambar-gambar ini:
- Daun dengan bintik-bintik, lesi, perubahan warna, layu, atau kelainan pertumbuhan
- Bagian tanaman yang menunjukkan gejala penyakit atau stres
- Bagian tanaman sehat yang menunjukkan pertumbuhan normal
- Foto jarak dekat (close-up) jaringan tanaman

### TOLAK gambar-gambar ini (set is_plant_related=false):
- Ornamen, dekorasi, tanaman hias tiruan, atau barang dekoratif
- Objek non-tanaman (batu, mainan, furnitur, peralatan, dll.)
- Gambar tanpa material tanaman yang terlihat
- Gambar buram/tidak dapat dikenali

## ANALISIS KOMPOSISI GAMBAR:
Anda HARUS menganalisis komposisi gambar untuk menentukan kebutuhan alat deteksi:

### image_shows_multiple_leaves:
- Set ke TRUE: Gambar menunjukkan 2 atau lebih daun dengan jelas (meskipun tumpang tindih)
- Set ke FALSE: Hanya satu daun, atau tidak ada daun yang terlihat

### image_shows_non_leaf_parts:
- Set ke TRUE: Gambar terutama menunjukkan buah, batang, bunga, polong, akar (bukan daun)
- Set ke FALSE: Gambar terutama menunjukkan daun atau gejala terkait daun

**Contoh:**
- Satu daun berpenyakit = FALSE untuk keduanya
- 3 daun dengan bintik = TRUE untuk multiple_leaves, FALSE untuk non_leaf
- Satu buah tomat = FALSE untuk multiple_leaves, TRUE untuk non_leaf
- Batang dengan lesi = FALSE untuk multiple_leaves, TRUE untuk non_leaf

## Tugas:
1. **Pertama**: Periksa gambar dengan teliti dan tentukan apakah terkait tanaman
2. **Kedua**: Analisis komposisi gambar (tunggal vs jamak, daun vs non-daun)
3. **Ketiga**: Jika terkait tanaman, buat pertanyaan pengguna seperti yang dijelaskan di bawah
4. **Keempat**: Jika TIDAK terkait tanaman, set is_plant_related=false dan gunakan nilai placeholder

Gambar menunjukkan: {symptoms}

## Buat pertanyaan pengguna (hanya jika terkait tanaman):
1. Sebutkan spesies tanaman ({plant})
2. Deskripsikan gejala yang terlihat dalam istilah samar dan non-teknis
3. Minta identifikasi dan saran penanganan

Gunakan bahasa Indonesia yang natural.
Contoh format: "{plant} saya memiliki [deskripsi samar]. Ini kira-kira kenapa ya dan bagaimana cara mengobatinya?" """


def get_direct_inquiry_prompt(plant: str, symptoms: str, image_url: str) -> str:
    """Prompt untuk skenario 2: Pertanyaan penyakit langsung (hanya spesies) + permintaan penanganan."""
    return f"""Anda adalah pemilik tanaman yang mengetahui spesies tanaman Anda dan menginginkan jawaban langsung.

## CEK KUALITAS KRITIS - BACA DENGAN TELITI:
Anda HARUS mengevaluasi apakah gambar tersebut benar-benar terkait dengan penyakit/kesehatan tanaman:

### TERIMA gambar-gambar ini:
- Daun dengan bintik, lesi, perubahan warna, layu, atau kelainan pertumbuhan
- Bagian tanaman yang menunjukkan gejala penyakit atau stres
- Bagian tanaman sehat

### TOLAK gambar-gambar ini (set is_plant_related=false):
- Ornamen, dekorasi, tanaman plastik
- Objek non-tanaman
- Gambar tanpa material tanaman

## ANALISIS KOMPOSISI GAMBAR:
### image_shows_multiple_leaves:
- TRUE: 2+ daun terlihat jelas
- FALSE: Daun tunggal atau tidak ada daun

### image_shows_non_leaf_parts:
- TRUE: Menunjukkan buah, batang, bunga, polong, akar
- FALSE: Terutama menunjukkan daun

## Tugas:
1. **Pertama**: Tentukan apakah gambar terkait tanaman
2. **Kedua**: Analisis komposisi gambar
3. **Ketiga**: Jika ya, buat pertanyaan langsung; jika tidak, set is_plant_related=false

Gambar menunjukkan: {symptoms}

## Buat pertanyaan pengguna (hanya jika terkait tanaman):
1. Penyebutan spesies tanaman dengan jelas ({plant})
2. **JANGAN** mendeskripsikan gejala visual yang terlihat di gambar
3. Permintaan langsung untuk identifikasi kondisi atau penyakit
4. Permintaan saran manajemen/pengobatan

Gunakan bahasa Indonesia yang natural.
Contoh format: "Saya punya tanaman {plant} yang daunnya seperti ini. Ini apakah penyakit dan bagaimana cara menanganinya?" """


def get_general_inquiry_prompt(plant: str, symptoms: str, image_url: str) -> str:
    """Prompt untuk skenario 3: Identifikasi umum (tanpa spesies/gejala) + permintaan penanganan."""
    return f"""Anda adalah pemilik tanaman yang mengambil foto dan ingin tahu apa yang salah.

## CEK KUALITAS KRITIS - BACA DENGAN TELITI:
Anda HARUS mengevaluasi apakah gambar tersebut benar-benar terkait dengan penyakit/kesehatan tanaman:

### TERIMA gambar-gambar ini:
- Daun dengan bintik, lesi, perubahan warna, layu, atau kelainan pertumbuhan
- Bagian tanaman yang menunjukkan gejala penyakit atau stres
- Bagian tanaman sehat

### TOLAK gambar-gambar ini (set is_plant_related=false):
- Ornamen, dekorasi, tanaman plastik
- Objek non-tanaman
- Gambar tanpa material tanaman

## ANALISIS KOMPOSISI GAMBAR (PENTING):
### image_shows_multiple_leaves:
- TRUE: 2+ daun terlihat jelas
- FALSE: Daun tunggal atau tidak ada daun

### image_shows_non_leaf_parts:
- TRUE: Menunjukkan buah, batang, bunga, polong, akar
- FALSE: Terutama menunjukkan daun

## Tugas:
1. **Pertama**: Tentukan apakah gambar terkait tanaman
2. **Kedua**: Analisis komposisi gambar
3. **Ketiga**: Jika ya, buat pertanyaan umum; jika tidak, set is_plant_related=false

Gambar menunjukkan: {symptoms}

## Buat pertanyaan pengguna (hanya jika terkait tanaman):
1. Pertanyaan umum tentang apa yang salah dengan tanaman tersebut
2. **JANGAN** menyebutkan spesies tanaman ({plant})
3. **JANGAN** mendeskripsikan gejala visual secara spesifik
4. Permintaan identifikasi masalah dan saran penanganan

Gunakan bahasa Indonesia yang natural.
Contoh format: "Tanaman saya kenapa ya? Apakah ini penyakit? Gimana cara mengatasinya?" atau "Kenapa tanaman saya jadi begini? Apa yang harus saya lakukan?" """


def get_healthy_prompt(plant: str, symptoms: str, image_url: str, scenario: int) -> str:
    """Prompt untuk skenario tanaman sehat."""
    base_prompt = f"""## 🔍 CEK KUALITAS KRITIS - BACA DENGAN TELITI:
Anda HARUS mengevaluasi apakah gambar benar-benar terkait dengan kesehatan tanaman:

### TERIMA gambar-gambar ini:
- Daun, batang, atau bagian tanaman yang terlihat sehat
- Warna hijau segar, struktur kokoh, pertumbuhan normal
- Tanaman yang menunjukkan pertumbuhan subur

### TOLAK gambar-gambar ini (set is_plant_related=false):
- Ornamen, dekorasi, tanaman plastik
- Objek non-tanaman
- Gambar tanpa material tanaman
- Tanaman yang menunjukkan gejala penyakit yang jelas

## ANALISIS KOMPOSISI GAMBAR (PENTING):
### image_shows_multiple_leaves:
- TRUE: 2+ daun terlihat jelas
- FALSE: Daun tunggal atau tidak ada daun

### image_shows_non_leaf_parts:
- TRUE: Menunjukkan buah, batang, bunga, polong, akar
- FALSE: Terutama menunjukkan daun

## Tugas:
1. **Pertama**: Tentukan apakah gambar menunjukkan material tanaman sehat
2. **Kedua**: Analisis komposisi gambar
3. **Ketiga**: Jika ya, buat pertanyaan; jika tidak, set is_plant_related=false

Gambar menunjukkan: {symptoms}

## Buat pertanyaan pengguna (hanya jika terkait tanaman):"""
    
    if scenario == 1:
        return base_prompt + f"""
1. Sebutkan spesies tanaman ({plant})
2. Deskripsikan tampilannya yang sehat (segar, hijau, dll.)
3. Minta konfirmasi bahwa tanaman tersebut sehat
4. Minta tips perawatan

Gunakan bahasa Indonesia yang natural.
Contoh format: "Daun {plant} saya ini terlihat hijau dan segar. Apakah ini tandanya pohon saya sehat? Ada tips untuk perawatannya?" """
    elif scenario == 2:
        return base_prompt + f"""
1. Sebutkan spesies tanaman ({plant})
2. **JANGAN** mendeskripsikan tampilan visualnya
3. Minta konfirmasi kesehatan
4. Minta saran perawatan

Gunakan bahasa Indonesia yang natural.
Contoh format: "Apakah tanaman {plant} saya ini terlihat sehat? Ada saran untuk perawatannya?" """
    else:
        return base_prompt + f"""
1. Pertanyaan umum tentang kesehatan tanaman
2. **JANGAN** menyebutkan spesies tanaman ({plant})
3. **JANGAN** mendeskripsikan tampilan visualnya
4. Minta tips perawatan

Gunakan bahasa Indonesia yang natural.
Contoh format: "Apakah tanaman saya sehat? Bagaimana cara merawatnya agar tumbuh dengan baik?" """

✅ Template prompt yang ditingkatkan dengan kontrol kualitas dan analisis komposisi gambar telah didefinisikan


In [49]:
# 5. Conditional Tool Call Logic
def get_disease_tool_calls(llm_response: Dict[str, Any] = None) -> List[Dict[str, Any]]:
    """
    Get tool calls for disease identification with conditional detection.
    
    Based on agent system prompt:
    - Mandatory: plant_disease_identification + knowledgebase_search
    - Conditional: closed_set_leaf_detection ONLY if multiple leaves visible
    - Conditional: open_vocabulary_plant_detection ONLY if showing multiple/complex non-leaf parts
    
    Do NOT use detection tools for:
    - Single leaf with clear focus
    - Single fruit/stem/flower with clear focus
    """
    base_tools = [
        {"name": "plant_disease_identification", "args": {}},
        {"name": "knowledgebase_search", "args": {}}
    ]
    
    # Add conditional detection based on LLM's image analysis
    if llm_response:
        # Use closed_set_leaf_detection only for multiple leaves
        if llm_response.get("image_shows_multiple_leaves", False):
            base_tools.insert(0, {"name": "closed_set_leaf_detection", "args": {}})
            print("  → Added closed_set_leaf_detection (multiple leaves detected)")
        
        # Use open_vocabulary_plant_detection only for non-leaf parts
        # AND only if it's not just a single clear organ
        elif llm_response.get("image_shows_non_leaf_parts", False):
            base_tools.insert(0, {"name": "open_vocabulary_plant_detection", "args": {}})
            print("  → Added open_vocabulary_plant_detection (non-leaf structures detected)")
        
        # Single leaf or single clear organ = no detection tool needed
        else:
            print("  → No detection tool needed (single clear plant part)")
    
    return base_tools


def get_healthy_tool_calls() -> List[Dict[str, Any]]:
    """Get tool calls for healthy plant scenarios."""
    return [{"name": "knowledgebase_search", "args": {}}]


def get_reference_goal_for_disease(user_text: str, plant: str, class_name: str) -> str:
    """Generate reference goal for disease scenarios."""
    return f"Lakukan triase visual pada gejala {plant} yang terlihat di gambar, gunakan alat identifikasi penyakit untuk mengonfirmasi {class_name}, lalu berikan panduan manajemen dan pengobatan berbasis bukti."


def get_reference_goal_for_healthy(user_text: str, plant: str) -> str:
    """Generate reference goal for healthy scenarios."""
    return f"Nilai status kesehatan {plant} dari gambar, konfirmasi bahwa tanaman sehat, dan berikan rekomendasi perawatan dan pemeliharaan."


In [50]:
# 6. Initialize Gemini Model
def initialize_gemini_model() -> ChatGoogleGenerativeAI:
    """Initialize the Gemini model for structured output."""
    api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
    
    if not api_key:
        print("Warning: No API key found in environment variables.")
        print("Set GOOGLE_API_KEY or GEMINI_API_KEY before running generation.")
        return None
    
    try:
        model = ChatGoogleGenerativeAI(
            model="gemini-3-flash-preview",
            temperature=0.7,
            max_tokens=2000
        )
        print("Gemini model initialized successfully")
        return model
    except Exception as e:
        print(f"Failed to initialize Gemini model: {e}")
        return None

# Test model initialization
model = initialize_gemini_model()
if model:
    print("Model ready for generation")
else:
    print("Model initialization failed - check API key")

Gemini model initialized successfully
Model ready for generation


In [51]:
# 7. Core VQA Generation Functions (WITH QUALITY CONTROL)
def generate_vqa_pair(
    model: ChatGoogleGenerativeAI,
    image_metadata: Dict[str, Any],
    reference_info: Dict[str, Any],
    scenario_type: str,
    scenario_num: int
) -> Optional[Dict[str, Any]]:
    """
    Generate a single VQA pair with quality control.
    
    Returns:
        Dictionary with VQA data or None if image is unrelated
    """
    try:
        # Extract information
        actual_class = image_metadata["class"]
        plant = extract_plant_from_class(actual_class)
        cause_field = image_metadata["cause"]
        status = image_metadata["status"]
        image_url = image_metadata["image_url"]
        filename = image_metadata["path"].split("\\")[-1]
        
        # Get reference info using the actual class name (e.g., 'apple black rot')
        ref_info = reference_lookup.get(actual_class, {})
        symptoms = ref_info.get("symptoms", image_metadata.get("caption", ""))
        management = ref_info.get("management", ref_info.get("maintenance", ""))
        
        # Determine prompt
        if scenario_type == "diseased":
            if scenario_num == 1:
                prompt_template = get_vague_symptoms_prompt(plant, symptoms, "")
                prompt_type = "vague_symptoms"
            elif scenario_num == 2:
                prompt_template = get_direct_inquiry_prompt(plant, symptoms, "")
                prompt_type = "direct_inquiry"
            else:
                prompt_template = get_general_inquiry_prompt(plant, symptoms, "")
                prompt_type = "general_inquiry"
        else:
            prompt_template = get_healthy_prompt(plant, symptoms, "", scenario_num)
            prompt_type = f"healthy_scenario_{scenario_num}"
        
        # Create multimodal message
        message = HumanMessage(
            content=[
                {"type": "text", "text": prompt_template},
                {"type": "image_url", "image_url": image_url},
            ]
        )
        
        # Generate with structured output
        structured_llm = model.with_structured_output(
            schema=VQAPair.model_json_schema(),
            method="json_schema"
        )
        
        result = structured_llm.invoke([message])
        
        # Check if result is valid
        if result is None:
            print(f"⚠️  SKIPPED: {filename} - Failed to generate structured output")
            return None
        
        # QUALITY CHECK: Filter out unrelated images
        if not result.get("is_plant_related", False):
            print(f"⚠️  FILTERED: {filename} - Not plant-related")
            return None
        
        # Build logic based on scenario
        if scenario_type == "diseased":
            tool_calls = get_disease_tool_calls(result)
            reference_goal = get_reference_goal_for_disease(result["user_text"], plant, actual_class)
            pathogen_type = extract_pathogen_type(status, cause_field)
            reference_context = f"Tanaman: {plant}. Penyakit: {actual_class}. Gejala: {symptoms}. Penanganan: {management}"
        else:
            tool_calls = get_healthy_tool_calls()
            reference_goal = get_reference_goal_for_healthy(result["user_text"], plant)
            pathogen_type = "healthy"
            reference_context = f"Tanaman: {plant}. Status: Sehat. Pemeliharaan: {management}"
        
        # Construct final entry
        entry = {
            "inputs": {
                "user_text": result["user_text"],
                "image_url": image_url
            },
            "outputs": {
                "reference_answer": result.get("reference_answer", management),
                "reference_goal": reference_goal,
                "reference_context": reference_context,
                "reference_tool_calls": tool_calls
            },
            "metadata": {
                "class": actual_class,
                "plant": plant,
                "pathogen_type": pathogen_type,
                "prompt_type": prompt_type,
                "filename": filename,
                "is_plant_related": True
            }
        }
        
        return entry
        
    except Exception as e:
        print(f"❌ Error saat membuat pasangan VQA for {filename}: {e}")
        return None


def generate_scenario_dataset(
    model: ChatGoogleGenerativeAI,
    metadata: Dict[str, Dict[str, Any]],
    scenario_type: str,
    scenario_num: int,
    target_size: int = None
) -> List[Dict[str, Any]]:
    """Generate dataset for a specific scenario with filtering."""
    # Filter by status
    filtered_items = {k: v for k, v in metadata.items() if v["status"] == scenario_type}
    
    # Group items by class
    items_by_class = {}
    for filename, item_data in filtered_items.items():
        class_name = item_data["class"]
        if class_name not in items_by_class:
            items_by_class[class_name] = []
        items_by_class[class_name].append((filename, item_data))
    
    # For each class, select 3 images and assign scenarios 1, 2, 3 respectively
    selected_items = []
    for class_name, items in items_by_class.items():
        # Assign scenario numbers cyclically based on index
        for idx, (filename, item_data) in enumerate(items):
            # We only want 3 scenarios per class max for the main generation
            if idx < 3:
                assigned_scenario = (idx % 3) + 1
                selected_items.append((filename, item_data, assigned_scenario))
    
    # Apply target_size if specified (for testing)
    if target_size:
        selected_items = selected_items[:target_size]
    
    print(f"Selected {len(selected_items)} images for {scenario_type} classes")
    
    # Generate with filtering
    results = []
    filtered_count = 0
    skipped_count = 0
    
    for idx, (filename, item_data, assigned_scenario) in enumerate(tqdm(selected_items, desc=f"Generating {scenario_type} scenarios")):
        # Check if already generated
        if filename in existing_results_lookup:
            results.append(existing_results_lookup[filename])
            skipped_count += 1
            continue
            
        result = generate_vqa_pair(model, item_data, reference_lookup.get(item_data["class"], {}), scenario_type, assigned_scenario)
        
        if result:
            results.append(result)
        else:
            filtered_count += 1
        
        import time
        time.sleep(0.05)
    
    print(f"✅ Generated {len(results) - skipped_count} new entries, reused {skipped_count} existing, (filtered {filtered_count})")
    return results

print("✅ VQA generation functions ready and cleaned up")


✅ VQA generation functions ready and cleaned up


In [52]:
# 8. Main Generation Function
def generate_all_scenarios():
    """Generate all scenarios: each class's 3 images get scenarios 1, 2, 3 respectively."""
    print("\n" + "="*60)
    print("GENERATING ALL SCENARIOS")
    print("Each class's 3 images: image1→scenario1, image2→scenario2, image3→scenario3")
    print("="*60)
    
    model = initialize_gemini_model()
    if not model:
        print("❌ Model not initialized")
        return None
    
    all_results = []
    
    # Generate for diseased classes
    print("\n[1/2] Generating diseased scenarios...")
    diseased_results = generate_scenario_dataset(model, metadata, "diseased", 1)  # scenario_num is ignored now
    all_results.extend(diseased_results)
    
    # Generate for healthy classes
    print("\n[2/2] Generating healthy scenarios...")
    healthy_results = generate_scenario_dataset(model, metadata, "healthy", 1)  # scenario_num is ignored now
    all_results.extend(healthy_results)
    
    # Save individual files for backward compatibility
    output_file = OUTPUT_DIR / "vqa_all_scenarios_id.json"
    with open(output_file, 'w') as f:
        json.dump(all_results, f, indent=2)
    
    print(f"✅ Berhasil membuat {len(all_results)} total entri")
    print(f"💾 Tersimpan: {output_file}")
    return all_results


In [53]:
# 9. Combine All Scenarios
def combine_all_datasets():
    """Combine all scenario datasets into one final dataset."""
    print("\n" + "="*60)
    print("MENGGABUNGKAN SEMUA SKENARIO")
    print("="*60)
    
    all_results = []
    
    # Check for the main output file first
    main_file = OUTPUT_DIR / "vqa_all_scenarios_id.json"
    if main_file.exists():
        with open(main_file, 'r') as f:
            all_results = json.load(f)
        print(f"✅ Berhasil memuat {len(all_results)} entri dari {main_file.name}")
    else:
        scenario_files = [
            OUTPUT_DIR / "vqa_scenario1_diseased.json",
            OUTPUT_DIR / "vqa_scenario2_diseased.json", 
            OUTPUT_DIR / "vqa_scenario3_diseased.json",
            OUTPUT_DIR / "vqa_healthy_scenarios.json"
        ]
        
        for file_path in scenario_files:
            if file_path.exists():
                with open(file_path, 'r') as f:
                    data = json.load(f)
                    all_results.extend(data)
                    print(f"✅ Berhasil memuat {len(data)} entri dari {file_path.name}")
            else:
                print(f"⚠️  File hilang: {file_path.name}")
    
    if all_results:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(all_results, f, indent=2)
        
        print(f"\n💾 Dataset gabungan disimpan ke: {OUTPUT_FILE}")
        print(f"📊 Total entri: {len(all_results)}")
        
        # Distribution
        by_status = {}
        by_prompt_type = {}
        for entry in all_results:
            status = "healthy" if entry["metadata"]["pathogen_type"] == "healthy" else "diseased"
            prompt_type = entry["metadata"]["prompt_type"]
            by_status[status] = by_status.get(status, 0) + 1
            by_prompt_type[prompt_type] = by_prompt_type.get(prompt_type, 0) + 1
        
        print(f"  Berdasarkan status: {by_status}")
        print(f"  Berdasarkan tipe prompt: {by_prompt_type}")
        
        # Expected calculation
        print(f"\n📋 Kalkulasi yang Diharapkan:")
        print(f"  Gambar Penyakit: {validation['disease_images']} × 1 skenario = {validation['disease_images']} entri")
        print(f"  Gambar Sehat: {validation['healthy_images']} × 1 skenario = {validation['healthy_images']} entri")
        print(f"  Total yang diharapkan: {validation['expected_total']} entri")
        print(f"  Aktual: {len(all_results)} entri")
        
        # Check if we got the expected amount
        if len(all_results) == validation['expected_total']:
            print(f"✅ Cocok sempurna! Semua {validation['expected_total']} entri berhasil dibuat.")
        else:
            print(f"⚠️  Ketidakcocokan: Mengharapkan {validation['expected_total']}, mendapatkan {len(all_results)}")
            print(f"   Hal ini mungkin disebabkan oleh penyaringan gambar yang tidak terkait tanaman oleh LLM.")
        
        return all_results
    else:
        print("❌ Data tidak ditemukan")
        return None


# 10. Main Execution
def main():
    """Generate all scenarios and combine."""
    print("Pembuatan Dataset VQA untuk Identifikasi Penyakit Tanaman")
    print("="*60)
    
    # Check API key
    api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
    if not api_key:
        print("❌ ERROR: API key tidak ditemukan!")
        print("Set GOOGLE_API_KEY atau GEMINI_API_KEY")
        return
    
    print("✅ API key ditemukan")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    print("\n📝 Kalkulasi Dataset:")
    print(f"  Input: {len(metadata)} entri metadata")
    print(f"  - Kelas Penyakit: {validation['disease_classes']} kelas")
    print(f"  - Kelas Sehat: {validation['healthy_classes']} kelas")
    print(f"  - Gambar Penyakit: {validation['disease_images']}")
    print(f"  - Gambar Sehat: {validation['healthy_images']}")
    print("\n  Output: 1 skenario per gambar (3 gambar per kelas = total 3 skenario)")
    print(f"  - Skenario Penyakit: {validation['disease_images']} × 1 = {validation['disease_images']} entri")
    print(f"  - Skenario Sehat: {validation['healthy_images']} × 1 = {validation['healthy_images']} entri")
    print(f"  - Total yang diharapkan: {validation['expected_total']} entri")
    
    response = input("\nLanjutkan pembuatan? (y/n): ")
    if response.lower() != 'y':
        print("❌ Pembuatan dibatalkan")
        return
    
    print("\n" + "="*60)
    print("MEMULAI PEMBUATAN")
    print("="*60)
    
    # Generate all scenarios
    print("\n[1/2] Membuat skenario penyakit...")
    diseased_results = generate_scenario_dataset(model, metadata, "diseased", 1)
    
    print("\n[2/2] Membuat skenario sehat...")
    healthy_results = generate_scenario_dataset(model, metadata, "healthy", 1)
    
    # Combine
    all_results = diseased_results + healthy_results
    
    # Save
    output_file = OUTPUT_DIR / "vqa_all_scenarios_id.json"
    with open(output_file, 'w') as f:
        json.dump(all_results, f, indent=2)
    
    print(f"\n💾 Berhasil menyimpan semua skenario ke: {output_file}")
    
    # Final combine
    print("\n[FINAL] Membuat dataset akhir...")
    final_dataset = combine_all_datasets()
    
    if final_dataset:
        print("\n" + "="*60)
        print("PEMBUATAN SELESAI!")
        print("="*60)
        print(f"Dataset akhir: {OUTPUT_FILE}")
        print(f"Total entri: {len(final_dataset)}")
    else:
        print("\n❌ Pembuatan gagal")

In [54]:
healthy_results = generate_scenario_dataset(model, metadata, "healthy", 1)

Selected 90 images for healthy classes


Generating healthy scenarios:   0%|          | 0/90 [00:00<?, ?it/s]

✅ Generated 1 new entries, reused 89 existing, (filtered 0)


In [55]:
diseased_results = generate_scenario_dataset(model, metadata, "diseased", 1)

Selected 177 images for diseased classes


Generating diseased scenarios:   0%|          | 0/177 [00:00<?, ?it/s]

✅ Generated 0 new entries, reused 177 existing, (filtered 0)


In [56]:
all_results = diseased_results + healthy_results

# Save
output_file = OUTPUT_DIR / "vqa_all_scenarios_id.json"
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\n💾 Berhasil menyimpan semua skenario ke: {output_file}")

# Final combine
print("\n[FINAL] Membuat dataset akhir...")
final_dataset = combine_all_datasets()


💾 Berhasil menyimpan semua skenario ke: data\vqa\vqa_all_scenarios_id.json

[FINAL] Membuat dataset akhir...

MENGGABUNGKAN SEMUA SKENARIO
✅ Berhasil memuat 267 entri dari vqa_all_scenarios_id.json

💾 Dataset gabungan disimpan ke: data\vqa\vqa_dataset.json
📊 Total entri: 267
  Berdasarkan status: {'diseased': 177, 'healthy': 90}
  Berdasarkan tipe prompt: {'vague_symptoms': 59, 'direct_inquiry': 59, 'general_inquiry': 59, 'healthy_scenario_1': 30, 'healthy_scenario_2': 30, 'healthy_scenario_3': 30}

📋 Kalkulasi yang Diharapkan:
  Gambar Penyakit: 177 × 1 skenario = 177 entri
  Gambar Sehat: 90 × 1 skenario = 90 entri
  Total yang diharapkan: 267 entri
  Aktual: 267 entri
✅ Cocok sempurna! Semua 267 entri berhasil dibuat.


In [57]:
# # 16. Jalankan menu interaktif
# print("\n" + "="*60)
# print("MENU UTAMA - PILIHAN CEPAT")
# print("="*60)

# # Uncomment baris di bawah untuk menjalankan menu interaktif
# # interactive_menu()

# print("\nPILIHAN:")
# print("1. Tampilkan status dataset")
# print("2. Bersihkan dataset dari entri yang tidak ada di metadata")
# print("3. Generate dataset baru (akan skip yang sudah ada)")
# print("4. Regenerasi gambar spesifik")
# print("5. Menu interaktif lengkap")

# choice = input("\nPilih opsi (1-5): ").strip()

# if choice == "1":
#     show_dataset_status()
# elif choice == "2":
#     clean_existing_dataset()
# elif choice == "3":
#     main()
# elif choice == "4":
#     print("\nMasukkan nama file gambar (pisahkan dengan koma jika banyak):")
#     files_input = input("File: ").strip()
#     if files_input:
#         file_list = [f.strip() for f in files_input.split(",")]
#         regenerate_for_images(file_list)
#     else:
#         print("❌ Tidak ada file yang dimasukkan")
# elif choice == "5":
#     interactive_menu()
# else:
#     print("❌ Pilihan tidak valid, coba lagi")

In [58]:
# Or generate all scenarios directly (without prompts)
# all_results = generate_all_scenarios()

In [59]:
# # Or combine existing datasets
# final_dataset = combine_all_datasets()
# print(f"Final dataset has {len(final_dataset)} entries")